# 📊 Embryoscope Embryo Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database and **AWS Athena (Production)** to execute data quality queries on the `embryoscope_embrioes` table. It retrieves:
1. Total row count
2. Unique embryo IDs
3. Unique patient IDs
4. Unique patient IDxs
5. Unique `prontuario` (medical record numbers)
6. Percentage of patients/records with a valid `prontuario` (not null or -1)

Metrics are calculated overall, per year (`embryo_EmbryoDate`), and per clinic location.

In [ ]:
import duckdb
import pandas as pd
from pyathena import connect

# Connections setup
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print("Connecting to AWS Athena...")
ath_con = connect(
    region_name="sa-east-1",
    work_group="datalake-admins"
)
ath_cur = ath_con.cursor()

## 🔍 Part 1: DuckDB Queries (Local Gold Layer)
Running local metrics on `gold.embryoscope_embrioes` including `patient_PatientIDx`.

In [ ]:
# 1. DuckDB Overall
duck_overall_q = """
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes;
"""
df_duck_overall = duck_con.execute(duck_overall_q).df()
print("--- DuckDB Overall Totals ---")
display(df_duck_overall)

# 2. DuckDB Per Year
duck_year_q = """
SELECT 
    EXTRACT(YEAR FROM embryo_EmbryoDate) as embryo_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
df_duck_year = duck_con.execute(duck_year_q).df()
print("\n--- DuckDB Per Year ---")
display(df_duck_year)

# 3. DuckDB Per Location
duck_loc_q = """
SELECT 
    patient_unit_huntington,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
df_duck_loc = duck_con.execute(duck_loc_q).df()
print("\n--- DuckDB Per Location ---")
display(df_duck_loc)

## ☁️ Part 2: AWS Athena Queries (Production Gold Layer)
Running metrics on `gold_huntington_prod.embryoscope_embrioes` including `patient_id_x`.

In [ ]:
# 1. Athena Overall
ath_overall_q = """
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes;
"""
ath_cur.execute(ath_overall_q)
df_ath_overall = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("--- Athena Overall Totals ---")
display(df_ath_overall)

# 2. Athena Per Year
ath_year_q = """
SELECT 
    EXTRACT(YEAR FROM embryo_embryo_date) as embryo_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
ath_cur.execute(ath_year_q)
df_ath_year = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("\n--- Athena Per Year ---")
display(df_ath_year)

# 3. Athena Per Location
ath_loc_q = """
SELECT 
    unit_huntington,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
ath_cur.execute(ath_loc_q)
df_ath_loc = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("\n--- Athena Per Location ---")
display(df_ath_loc)

## ⚖️ Part 3: Comparative Breakdown (DuckDB vs Athena Prod)
Below, we merge the results to highlight discrepancies, now containing unique Patient IDx counts.

In [ ]:
# 1. Location Comparison
df_duck_loc_clean = df_duck_loc.rename(columns={
    'patient_unit_huntington': 'location',
    'total_rows': 'duck_rows',
    'unique_patients': 'duck_patients',
    'unique_patient_idxs': 'duck_patient_idxs',
    'unique_prontuarios': 'duck_prontuarios',
    'pct_patients_with_valid_prontuario': 'duck_valid_pront_pct'
})[['location', 'duck_rows', 'duck_patients', 'duck_patient_idxs', 'duck_prontuarios', 'duck_valid_pront_pct']]

df_ath_loc_clean = df_ath_loc.rename(columns={
    'unit_huntington': 'location',
    'total_rows': 'ath_rows',
    'unique_patients': 'ath_patients',
    'unique_patient_idxs': 'ath_patient_idxs',
    'unique_prontuarios': 'ath_prontuarios',
    'pct_patients_with_valid_prontuario': 'ath_valid_pront_pct'
})[['location', 'ath_rows', 'ath_patients', 'ath_patient_idxs', 'ath_prontuarios', 'ath_valid_pront_pct']]

df_comp_loc = pd.merge(df_duck_loc_clean, df_ath_loc_clean, on='location')
print("--- Side-by-side Location Comparison ---")
display(df_comp_loc)

# 2. Year Comparison
df_duck_yr_clean = df_duck_year.rename(columns={
    'embryo_year': 'year',
    'total_rows': 'duck_rows',
    'unique_patients': 'duck_patients',
    'unique_patient_idxs': 'duck_patient_idxs',
    'unique_prontuarios': 'duck_prontuarios',
    'pct_patients_with_valid_prontuario': 'duck_valid_pront_pct'
})[['year', 'duck_rows', 'duck_patients', 'duck_patient_idxs', 'duck_prontuarios', 'duck_valid_pront_pct']]

df_ath_yr_clean = df_ath_year.rename(columns={
    'embryo_year': 'year',
    'total_rows': 'ath_rows',
    'unique_patients': 'ath_patients',
    'unique_patient_idxs': 'ath_patient_idxs',
    'unique_prontuarios': 'ath_prontuarios',
    'pct_patients_with_valid_prontuario': 'ath_valid_pront_pct'
})[['year', 'ath_rows', 'ath_patients', 'ath_patient_idxs', 'ath_prontuarios', 'ath_valid_pront_pct']]

# Ensure both are integers for seamless merge
df_duck_yr_clean['year'] = df_duck_yr_clean['year'].astype(float).astype(int)
df_ath_yr_clean['year'] = df_ath_yr_clean['year'].astype(float).astype(int)

df_comp_yr = pd.merge(df_duck_yr_clean, df_ath_yr_clean, on='year')
print("\n--- Side-by-side Year Comparison ---")
display(df_comp_yr)

## 🔍 Part 4: Discrepant Patient Sample Comparison (First Record)
This section finds 5 patients who have a valid `prontuario` in DuckDB, but have a missing/default (`NULL` or `-1`) `prontuario` in AWS Athena. It retrieves and displays the first record (line) for each of these 5 patients from both sources for direct comparison.

In [ ]:
# 1. Fetch unique patients (at patient_PatientIDx level) and their prontuarios
df_duck_pts = duck_con.execute("""
    SELECT DISTINCT patient_PatientIDx, patient_PatientID, patient_FirstName as first_name, prontuario
    FROM gold.embryoscope_embrioes
    WHERE prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1'
""").df()

ath_cur.execute("""
    SELECT DISTINCT patient_id_x, patient_id, first_name, prontuario
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_pts = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])

# Convert IDxs and IDs to string to prevent merge type mismatch
df_duck_pts['patient_PatientIDx'] = df_duck_pts['patient_PatientIDx'].astype(str)
df_ath_pts['patient_id_x'] = df_ath_pts['patient_id_x'].astype(str)

# 2. Find discrepancies where Athena prontuario is NULL or -1, joining on patient_id_x
merged_pts = pd.merge(df_duck_pts, df_ath_pts, left_on='patient_PatientIDx', right_on='patient_id_x', suffixes=('_duck', '_ath'))
discrepant = merged_pts[
    merged_pts['prontuario_ath'].isna() | 
    (merged_pts['prontuario_ath'] == -1) | 
    (merged_pts['prontuario_ath'].astype(str) == '-1') | 
    (merged_pts['prontuario_ath'].astype(str) == 'None')
].drop_duplicates(subset=['patient_PatientIDx']).head(5)

sample_patient_idxs = discrepant['patient_PatientIDx'].tolist()
print("Discrepant Patient IDxs identified:", sample_patient_idxs)

# 3. Fetch all records for only these 5 patients
df_duck_all_records = duck_con.execute(f"""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate
    FROM gold.embryoscope_embrioes
    WHERE CAST(patient_PatientIDx AS VARCHAR) IN ({','.join("'" + px + "'" for px in sample_patient_idxs)})
""").df()
df_duck_all_records['patient_PatientIDx'] = df_duck_all_records['patient_PatientIDx'].astype(str)

ath_cur.execute(f"""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date
    FROM gold_huntington_prod.embryoscope_embrioes
    WHERE patient_id_x IN ({','.join("'" + px + "'" for px in sample_patient_idxs)})
""")
df_ath_all_records = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_all_records['patient_id_x'] = df_ath_all_records['patient_id_x'].astype(str)

# 4. Display the first record for each patient in each database (sorted by patient_id_x)
df_duck_first = df_duck_all_records.drop_duplicates(subset=['patient_PatientIDx']).sort_values(by='patient_PatientIDx')
df_ath_first = df_ath_all_records.drop_duplicates(subset=['patient_id_x']).sort_values(by='patient_id_x')

print("\n🔹 [DuckDB] First line for each of the 5 discrepant patients (Sorted):")
display(df_duck_first)

print("\n🔸 [Athena Prod] First line for each of the 5 discrepant patients (Sorted):")
display(df_ath_first)

# 5. Query spouse names from Athena silver_clinisys_prod.view_pacientes using DuckDB's valid prontuarios
print("\n🌸 [Athena silver_clinisys_prod] Wife & Husband names matching DuckDB prontuarios:")
sample_prontuarios = df_duck_first['prontuario'].dropna().tolist()
sample_prontuarios_clean = [int(p) for p in sample_prontuarios if str(p) != '-1' and pd.notnull(p)]

if sample_prontuarios_clean:
    pront_list_str = ','.join(str(p) for p in sample_prontuarios_clean)
    clinisys_q = f"""
        SELECT codigo as prontuario, esposa_nome, marido_nome 
        FROM silver_clinisys_prod.view_pacientes 
        WHERE codigo IN ({pront_list_str})
    """
    ath_cur.execute(clinisys_q)
    df_clinisys_names = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    df_clinisys_names['prontuario'] = df_clinisys_names['prontuario'].astype(str)
    display(df_clinisys_names)
else:
    print("No valid prontuario values available to query Clinisys spouse names.")

## 📑 Part 5: Discrepancy & Exclusive Row Analysis
This section calculates the total count of discrepant embryo records (`embryo_EmbryoID` / `embryo_embryo_id`) that exist exclusively in one database but not the other, applying the sync lag and server outage filters. It displays the total discrepancy volumes and a sample of the exclusive records.

In [ ]:
# 1. Fetch all records from both databases with the reporting column set
print("Fetching all records from DuckDB...")
df_duck_all = duck_con.execute("""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate
    FROM gold.embryoscope_embrioes
""").df()
df_duck_all['embryo_EmbryoID'] = df_duck_all['embryo_EmbryoID'].astype(str)

print("Fetching all records from AWS Athena Prod...")
ath_cur.execute("""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_all = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_all['embryo_embryo_id'] = df_ath_all['embryo_embryo_id'].astype(str)

# Get max dates for sync lag verification
max_ath_date = pd.to_datetime(df_ath_all['embryo_embryo_date']).max()
max_duck_date = pd.to_datetime(df_duck_all['embryo_EmbryoDate']).max()

# Convert to datetime for filtering
df_duck_all['datetime_EmbryoDate'] = pd.to_datetime(df_duck_all['embryo_EmbryoDate'])
df_ath_all['datetime_embryo_date'] = pd.to_datetime(df_ath_all['embryo_embryo_date'])

# Print active filters
print("\n⚠️ Discrepancy comparison filters applied:")
print("  - Ignored Ibirapuera records from 2025 (known server issue)")
print("  - Filtered out records at or after the max embryo date of the target database (sync lag buffer)")
print(f"  - DuckDB max embryo date: {max_duck_date.strftime('%Y-%m-%d') if pd.notnull(max_duck_date) else 'N/A'}")
print(f"  - AWS Athena max embryo date: {max_ath_date.strftime('%Y-%m-%d') if pd.notnull(max_ath_date) else 'N/A'}")

# 2. Find records ONLY in DuckDB (missing in Athena)
df_only_duck = df_duck_all[~df_duck_all['embryo_EmbryoID'].isin(df_ath_all['embryo_embryo_id'])].copy()
df_only_duck = df_only_duck[~(
    (df_only_duck['patient_unit_huntington'] == 'Ibirapuera') &
    (df_only_duck['datetime_EmbryoDate'].dt.year == 2025)
)]
df_only_duck = df_only_duck[df_only_duck['datetime_EmbryoDate'] < max_ath_date]

print(f"\n🔴 Total records only in DuckDB: {len(df_only_duck)}")
if not df_only_duck.empty:
    print("Sample of records only in DuckDB (Filtered):")
    display(df_only_duck.head(5))
else:
    print("No exclusive DuckDB records found matching the restrictions.")

# 3. Find records ONLY in AWS Athena Prod (missing in DuckDB)
df_only_ath = df_ath_all[~df_ath_all['embryo_embryo_id'].isin(df_duck_all['embryo_EmbryoID'])].copy()
df_only_ath = df_only_ath[~(
    (df_only_ath['unit_huntington'] == 'Ibirapuera') &
    (df_only_ath['datetime_embryo_date'].dt.year == 2025)
)]
df_only_ath = df_only_ath[df_only_ath['datetime_embryo_date'] < max_duck_date]

print(f"\n🔵 Total records only in AWS Athena Prod: {len(df_only_ath)}")
if not df_only_ath.empty:
    print("Sample of records only in AWS Athena Prod (Filtered):")
    display(df_only_ath.head(5))
else:
    print("No exclusive Athena records found matching the restrictions.")

In [ ]:
df_only_ath[df_only_ath['patient_id'].str.contains('test')].shape

In [ ]:
df_only_ath[df_only_ath['unit_huntington']=='Ibirapuera']

## 🥚 Part 6: Embryo Number Mismatch Analysis
This section identifies embryo records that exist in **both** databases (matched by embryo ID) but have **different** values for the embryo number field (`embryo_embryo_number` in DuckDB vs. `embryo_number` in AWS Athena).

In [ ]:
# 1. Fetch reporting column set plus embryo numbers from both databases
print("Fetching embryo records with embryo numbers from DuckDB...")
df_duck_num = duck_con.execute("""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate, embryo_embryo_number
    FROM gold.embryoscope_embrioes
""").df()
df_duck_num['embryo_EmbryoID'] = df_duck_num['embryo_EmbryoID'].astype(str)

print("Fetching embryo records with embryo numbers from AWS Athena Prod...")
ath_cur.execute("""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date, embryo_number
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_num = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_num['embryo_embryo_id'] = df_ath_num['embryo_embryo_id'].astype(str)

# 2. Join the databases on embryo ID (inner join to find records existing in both)
df_merged_num = pd.merge(
    df_duck_num, 
    df_ath_num, 
    left_on='embryo_EmbryoID', 
    right_on='embryo_embryo_id', 
    suffixes=('_duck', '_ath')
)

# 3. Clean and convert embryo numbers to numeric to ensure clean comparison
df_merged_num['embryo_num_clean_duck'] = pd.to_numeric(df_merged_num['embryo_embryo_number'], errors='coerce')
df_merged_num['embryo_num_clean_ath'] = pd.to_numeric(df_merged_num['embryo_number'], errors='coerce')

# 4. Find records where clean embryo numbers do not match
df_mismatched_num = df_merged_num[
    df_merged_num['embryo_num_clean_duck'] != df_merged_num['embryo_num_clean_ath']
]

print(f"\n⚠️ Total matched embryo records with differing embryo numbers: {len(df_mismatched_num)}")
if not df_mismatched_num.empty:
    print("Sample of mismatched embryo number records:")
    # Display clean selection showing comparative columns
    display_cols = [
        'embryo_EmbryoID', 'patient_PatientIDx', 'first_name_duck', 
        'embryo_embryo_number', 'embryo_number', 
        'patient_unit_huntington', 'unit_huntington'
    ]
    display(df_mismatched_num[display_cols].head(10))
else:
    print("No embryo number discrepancies found for overlapping embryo records.")

## 🔬 Part 6: Morphological & Timing Annotations Completeness
Compares the non-null filling rates (%) of key embryology morphokinetics (`t2`, `t5`, `t8`, `tB`, `ICM`, `TE`) between local DuckDB and AWS Athena Prod.

In [ ]:
duck_ann = duck_con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        ROUND(COUNT(embryo_Time_t2)*100.0/COUNT(*), 2) as t2_pct,
        ROUND(COUNT(embryo_Time_t5)*100.0/COUNT(*), 2) as t5_pct,
        ROUND(COUNT(embryo_Time_t8)*100.0/COUNT(*), 2) as t8_pct,
        ROUND(COUNT(embryo_Time_tB)*100.0/COUNT(*), 2) as tb_pct,
        ROUND(COUNT(embryo_Value_ICM)*100.0/COUNT(*), 2) as icm_pct,
        ROUND(COUNT(embryo_Value_TE)*100.0/COUNT(*), 2) as te_pct
    FROM gold.embryoscope_embrioes
""").df()

ath_cur.execute("""
    SELECT 
        COUNT(*) as total_rows,
        ROUND(COUNT(embryo_time_t2)*100.0/COUNT(*), 2) as t2_pct,
        ROUND(COUNT(embryo_time_t5)*100.0/COUNT(*), 2) as t5_pct,
        ROUND(COUNT(embryo_time_t8)*100.0/COUNT(*), 2) as t8_pct,
        ROUND(COUNT(embryo_time_tb)*100.0/COUNT(*), 2) as tb_pct,
        ROUND(COUNT(embryo_value_icm)*100.0/COUNT(*), 2) as icm_pct,
        ROUND(COUNT(embryo_value_te)*100.0/COUNT(*), 2) as te_pct
    FROM gold_huntington_prod.embryoscope_embrioes
""")
ath_ann = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

df_ann_comp = pd.DataFrame({
    'annotation_field': ['t2_pct', 't5_pct', 't8_pct', 'tb_pct', 'icm_pct', 'te_pct'],
    'duck_fill_pct': [duck_ann[c].iloc[0] for c in ['t2_pct', 't5_pct', 't8_pct', 'tb_pct', 'icm_pct', 'te_pct']],
    'ath_fill_pct': [ath_ann[c].iloc[0] for c in ['t2_pct', 't5_pct', 't8_pct', 'tb_pct', 'icm_pct', 'te_pct']]
})
df_ann_comp['diff_pct'] = (df_ann_comp['duck_fill_pct'] - df_ann_comp['ath_fill_pct']).round(2)
print("--- Annotations Completion Rate Comparison (%) ---")
display(df_ann_comp)

## ❄️ Part 7: Embryo Fate Categorical Breakdown
Compares row distributions per `embryo_EmbryoFate` categorical value between local DuckDB and AWS Athena Prod.

In [ ]:
duck_fate = duck_con.execute("SELECT embryo_EmbryoFate as fate, COUNT(*) as duck_count FROM gold.embryoscope_embrioes GROUP BY 1").df()
ath_cur.execute("SELECT embryo_embryo_fate as fate, COUNT(*) as ath_count FROM gold_huntington_prod.embryoscope_embrioes GROUP BY 1")
ath_fate = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

merged_fate = pd.merge(duck_fate, ath_fate, on='fate', how='outer').fillna(0)
merged_fate['duck_count'] = merged_fate['duck_count'].astype(int)
merged_fate['ath_count'] = merged_fate['ath_count'].astype(int)
merged_fate['diff_count'] = merged_fate['duck_count'] - merged_fate['ath_count']
print("--- Embryo Fate Distribution Comparison ---")
display(merged_fate.sort_values(by='duck_count', ascending=False))

## 📅 Part 8: Monthly Volume Trend (`YYYY-MM`)
Displays monthly row counts for the recent 12 months to highlight exact sync lag thresholds.

In [ ]:
duck_month = duck_con.execute("SELECT strftime(embryo_EmbryoDate, '%Y-%m') as year_month, COUNT(*) as duck_rows FROM gold.embryoscope_embrioes GROUP BY 1 ORDER BY 1").df()
ath_cur.execute("SELECT date_format(embryo_embryo_date, '%Y-%m') as year_month, COUNT(*) as ath_rows FROM gold_huntington_prod.embryoscope_embrioes GROUP BY 1 ORDER BY 1")
ath_month = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

merged_month = pd.merge(duck_month, ath_month, on='year_month', how='outer').fillna(0)
merged_month['duck_rows'] = merged_month['duck_rows'].astype(int)
merged_month['ath_rows'] = merged_month['ath_rows'].astype(int)
merged_month['diff_rows'] = merged_month['duck_rows'] - merged_month['ath_rows']

print("--- Recent 12-Month Volume Trend ---")
display(merged_month.sort_values(by='year_month', ascending=False).head(12))

## 🧮 Part 9: Numeric Score & Age Reconciliation (`AgeAtFertilization`)
Reconciles `AgeAtFertilization` numeric values across matched embryo IDs in both databases to verify float precision.

In [ ]:
duck_age = duck_con.execute("SELECT embryo_EmbryoID, AgeAtFertilization FROM gold.embryoscope_embrioes WHERE AgeAtFertilization IS NOT NULL").df()
ath_cur.execute("SELECT embryo_embryo_id, age_at_fertilization FROM gold_huntington_prod.embryoscope_embrioes WHERE age_at_fertilization IS NOT NULL")
ath_age = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

duck_age['embryo_EmbryoID'] = duck_age['embryo_EmbryoID'].astype(str)
ath_age['embryo_embryo_id'] = ath_age['embryo_embryo_id'].astype(str)

merged_age = pd.merge(duck_age, ath_age, left_on='embryo_EmbryoID', right_on='embryo_embryo_id')
merged_age['abs_diff'] = (merged_age['AgeAtFertilization'] - merged_age['age_at_fertilization']).abs()
exact_matches = (merged_age['abs_diff'] < 0.01).sum()
total_matched = len(merged_age)

print(f"Total Matched Embryos Audited: {total_matched}")
print(f"Exact Age Matches (<0.01 diff): {exact_matches} ({exact_matches*100.0/total_matched:.2f}%)")
if exact_matches < total_matched:
    print("\nSample of numeric age discrepancies:")
    display(merged_age[merged_age['abs_diff'] >= 0.01].head(5))

## 🔗 Part 10: Patient ID Mapping Integrity (`patient_id` vs `patient_id_x`)
Audits the number of unique `patient_id_x` treatment records per `patient_id`.

In [ ]:
duck_integrity = duck_con.execute("SELECT patient_PatientID as patient_id, COUNT(DISTINCT patient_PatientIDx) as idx_count_duck FROM gold.embryoscope_embrioes GROUP BY 1").df()
ath_cur.execute("SELECT patient_id, COUNT(DISTINCT patient_id_x) as idx_count_ath FROM gold_huntington_prod.embryoscope_embrioes GROUP BY 1")
ath_integrity = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

duck_integrity['patient_id'] = duck_integrity['patient_id'].astype(str)
ath_integrity['patient_id'] = ath_integrity['patient_id'].astype(str)

merged_integrity = pd.merge(duck_integrity, ath_integrity, on='patient_id', how='outer').fillna(0)
merged_integrity['idx_count_duck'] = merged_integrity['idx_count_duck'].astype(int)
merged_integrity['idx_count_ath'] = merged_integrity['idx_count_ath'].astype(int)

print("Top patients with multiple patient_id_x treatment references in DuckDB:")
display(merged_integrity.sort_values(by='idx_count_duck', ascending=False).head(10))

In [ ]:
# Close database connections
print("Closing database connections...")
try:
    duck_con.close()
    print("DuckDB connection closed.")
except Exception as e:
    print(f"Error closing DuckDB connection: {e}")

try:
    ath_con.close()
    print("Athena connection closed.")
except Exception as e:
    print(f"Error closing Athena connection: {e}")